In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from tqdm import tqdm

sys.path.append(os.path.abspath("../../")) ; from EPF import variables
sys.path.insert(0, "../helpers/"); from run_parrellel import run_parallel

import lightgbm as lgb
import json
import joblib

### High-accuracy NEM price forecaster.

#### Method

Spike-aware direct multi-horizon LightGBM ensemble with seven components
per forecast horizon:

  1. BASE L1 model   – regression_l1 on arcsinh(clip(y, p97)). Clipping the
                       target at the 97th percentile removes extreme-spike
                       noise from the base model's loss, giving it a clean
                       training signal for the majority of intervals.

  2. BASE L2 model   – regression (MSE) on arcsinh(y), uncapped target.
                       Blending L1+L2 (Lago et al. 2021) reduces both MAE
                       and RMSE simultaneously. Per-horizon blend weight α
                       tuned on the validation set.

  3. SPIKE classifier – binary LightGBM estimating P(price > SPIKE_THRESHOLD).
                        Trained with scale_pos_weight to handle class imbalance.

  4. SPIKE regressor  – regression_l1 on arcsinh(y) (full range, no clipping).
                        Spike rows receive 10x sample weight.

  5. SPIKE quantile   – Predicts the 90th percentile for conservative spike ceilings. 
                        P90 quantile regression on arcsinh(y) for a
                        conservative spike ceiling estimate.

  6. DIP classifier   – binary LightGBM estimating P(price < SPIKE_THRESHOLD).
                        Trained with scale_pos_weight to handle class imbalance.

  7. DIP regressor    – regression_l1 on arcsinh(y) (full range, no clipping).
                        DIP rows receive 7x sample weight.

  8. DIP quantile   –   Predicts the 10th percentile for conservative dip floors. 
                        P10 quantile regression on arcsinh(y) for a
                        conservative dip floor estimate.



Inputs

In [2]:
y_train_individual_clipped_transformed = pd.read_parquet("Data/1_derived_targets/y_train_individual_clipped_transformed.parquet")
y_validation_individual_clipped_transformed = pd.read_parquet("Data/1_derived_targets/y_validation_individual_clipped_transformed.parquet")
y_train_individual_unclipped_transformed = pd.read_parquet("Data/1_derived_targets/y_train_individual_unclipped_transformed.parquet")
y_validation_individual_unclipped_transformed = pd.read_parquet("Data/1_derived_targets/y_validation_individual_unclipped_transformed.parquet")
positive_spike_labels_train = pd.read_parquet("Data/1_derived_targets/positive_spike_labels_train.parquet")
positive_spike_labels_validate = pd.read_parquet("Data/1_derived_targets/positive_spike_labels_validate.parquet")
negative_spike_labels_train = pd.read_parquet("Data/1_derived_targets/negative_spike_labels_train.parquet")
negative_spike_labels_validate = pd.read_parquet("Data/1_derived_targets/negative_spike_labels_validate.parquet")
y_train_loss_weights = pd.read_parquet("Data/1_derived_targets/y_train_loss_weights.parquet")
y_train_full_range_loss_weights = pd.read_parquet("Data/1_derived_targets/y_train_full_range_loss_weights.parquet")
y_train_positive_spike_loss_weights = pd.read_parquet("Data/1_derived_targets/y_train_positive_spike_loss_weights.parquet")
y_train_negative_spike_loss_weights = pd.read_parquet("Data/1_derived_targets/y_train_negative_spike_loss_weights.parquet")

In [3]:
# Load hyperparameters
with open("Data/2_hyper_parameters/hyperparameters.json", "r", encoding="utf-8") as f:
    MODEL_HYPERPARAMETERS = json.load(f)

full_range_regressor_clipped_MAE_loss_params = MODEL_HYPERPARAMETERS["full_range_regressor_clipped_MAE_loss"]
full_range_regressor_unclipped_RMSE_loss_params = MODEL_HYPERPARAMETERS["full_range_regressor_unclipped_RMSE_loss"]
positive_spike_classifier_binary_loss_params = MODEL_HYPERPARAMETERS["positive_spike_classifier_binary_loss"]
positive_spike_regressor_unclipped_mae_loss_params = MODEL_HYPERPARAMETERS["positive_spike_regressor_unclipped_mae_loss"]
positive_spike_regressor_unclipped_quantile_loss_params = MODEL_HYPERPARAMETERS["positive_spike_regressor_unclipped_quantile_loss"]
negative_spike_classifer_unclipped_binary_loss_params = MODEL_HYPERPARAMETERS["negative_spike_classifer_unclipped_binary_loss"]
negative_spike_regressor_unclipped_mae_loss_params = MODEL_HYPERPARAMETERS["negative_spike_regressor_unclipped_mae_loss"]
negative_spike_regressor_unclipped_quantile_loss_params = MODEL_HYPERPARAMETERS["negative_spike_regressor_unclipped_quantile_loss"]

Core logic

In [4]:
# LGBM model setup

def _train_model(item: dict):
    horizon = item["horizon"]
    model_name = item["model_name"]
    hyperparameters = item["hyperparameters"]
    X_train = item["X_train"]
    y_train = item["y_train"]
    X_validate = item["X_validate"]
    y_validate = item["y_validate"]
    loss_weights = item["loss_weights"]

    EARLY_STOPPING_ROUNDS = 75
    limiter = 10
    model = lgb.LGBMRegressor(**hyperparameters)
    
    model.fit(
        X_train.iloc[:limiter],
        y_train.iloc[:limiter],
        sample_weight=loss_weights.iloc[:limiter],
        eval_set=[(X_validate.iloc[:limiter], y_validate.iloc[:limiter])],
        callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
    )

    return {"horizon": horizon, "model_name": model_name, "model": model}

In [5]:
model_specs = [
    ("full_range_regressor_clipped_MAE_loss", full_range_regressor_clipped_MAE_loss_params, y_train_individual_clipped_transformed, y_validation_individual_clipped_transformed, y_train_full_range_loss_weights),
    ("full_range_regressor_unclipped_RMSE_loss", full_range_regressor_unclipped_RMSE_loss_params, y_train_individual_unclipped_transformed, y_validation_individual_unclipped_transformed, y_train_full_range_loss_weights),
    ("positive_spike_classifier_binary_loss", positive_spike_classifier_binary_loss_params, positive_spike_labels_train, positive_spike_labels_validate, y_train_loss_weights),
    ("positive_spike_regressor_unclipped_mae_loss", positive_spike_regressor_unclipped_mae_loss_params, y_train_individual_unclipped_transformed, y_validation_individual_unclipped_transformed, y_train_full_range_loss_weights),
    ("positive_spike_regressor_unclipped_quantile_loss", positive_spike_regressor_unclipped_quantile_loss_params, y_train_individual_unclipped_transformed, y_validation_individual_unclipped_transformed, y_train_positive_spike_loss_weights),
    ("negative_spike_classifer_unclipped_binary_loss", negative_spike_classifer_unclipped_binary_loss_params, negative_spike_labels_train, negative_spike_labels_validate, y_train_loss_weights),
    ("negative_spike_regressor_unclipped_mae_loss", negative_spike_regressor_unclipped_mae_loss_params, y_train_individual_unclipped_transformed, y_validation_individual_unclipped_transformed, y_train_negative_spike_loss_weights),
    ("negative_spike_regressor_unclipped_quantile_loss", negative_spike_regressor_unclipped_quantile_loss_params, y_train_individual_unclipped_transformed, y_validation_individual_unclipped_transformed, y_train_negative_spike_loss_weights),
]


In [6]:
feature_data = pd.read_parquet(variables.FEATURES_DATASET_PATH)

In [7]:
feature_data[:-5]

,nsw_price_asinh_lag_1,nsw_price_asinh_lag_2,nsw_price_asinh_lag_4,nsw_price_asinh_lag_12,nsw_price_asinh_lag_48,nsw_price_asinh_lag_96,nsw_price_asinh_lag_336,nsw_price_asinh_lag_335,nsw_price_asinh_lag_337,nsw_price_asinh_rmean_48,...,sa_price_vs_vic_price_spread,sa_price_vs_vic_price_spread_lag1,vic_price_vs_nsw_price_spread,vic_price_vs_nsw_price_spread_lag1,vic_price_vs_qld_price_spread,vic_price_vs_qld_price_spread_lag1,vic_price_vs_sa_price_spread,vic_price_vs_sa_price_spread_lag1,multi_region_spike,region_spike_count
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,12.968930,NaN,0.399640,NaN,10.339350,NaN,-12.968930,NaN,0.0,0.0
2018-01-01 00:10:00,0.810427,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,15.080300,12.968930,1.196190,0.399640,11.636780,10.339350,-15.080300,-12.968930,0.0,0.0
2018-01-01 00:15:00,0.849487,0.810427,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,16.358509,15.080300,1.050520,1.196190,12.244390,11.636780,-16.358509,-15.080300,0.0,0.0
2018-01-01 00:20:00,0.854923,0.849487,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,13.927330,16.358509,0.324450,1.050520,11.096110,12.244390,-13.927330,-16.358509,0.0,0.0
2018-01-01 00:25:00,0.827334,0.854923,0.810427,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,15.698750,13.927330,0.333230,0.324450,11.124200,11.096110,-15.698750,-13.927330,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-31 23:15:00,0.663137,0.545535,0.543191,0.645833,0.640081,0.541019,0.726974,0.773916,0.782782,0.619180,...,-0.180610,-0.150640,-56.020000,-70.201897,-62.450001,-77.750160,0.180610,0.150640,0.0,0.0
2025-12-31 23:20:00,0.545535,0.663137,0.545535,0.655941,0.718820,0.605098,0.773916,0.805145,0.726974,0.615569,...,-0.347180,-0.180610,-54.840000,-56.020000,-60.809448,-62.450001,0.347180,0.180610,0.0,0.0
2025-12-31 23:25:00,0.545535,0.545535,0.545535,0.559808,0.688672,0.510393,0.805145,0.786218,0.773916,0.612587,...,-1.758460,-0.347180,-44.389999,-54.840000,-49.551670,-60.809448,1.758460,0.347180,0.0,0.0


In [8]:
X_train = feature_data[(feature_data.index >= variables.TRAIN_START) & (feature_data.index <= variables.VALID_START)]
X_validate = feature_data[(feature_data.index > variables.VALID_START) & (feature_data.index <= variables.TEST_START)]
X_test = feature_data[feature_data.index > variables.TEST_START]

features_optimal_amount = pd.read_parquet(variables.FEATURES_OPTIMAL_AMOUNT_PATH)

In [9]:
display(feature_data.shape)
display(X_train.shape)
display(X_validate.shape)
display(X_test.shape)

(841536, 634)

(578305, 634)

(52992, 634)

(105120, 634)

In [10]:
features_optimal_amount[:10]

,feature,h1,h2,h3,h4,h5,h6,h7,h8,h9,...,h87,h88,h89,h90,h91,h92,h93,h94,h95,h96
0,nsw_price_asinh_lag_1,False,False,False,False,False,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False
1,nsw_price_asinh_lag_2,False,False,False,False,False,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False
2,nsw_price_asinh_lag_4,False,False,False,False,False,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False
3,nsw_price_asinh_lag_12,False,False,False,False,False,True,True,True,False,...,False,False,False,False,True,False,False,False,False,False
4,nsw_price_asinh_lag_48,True,True,True,True,False,True,True,True,False,...,False,False,False,False,True,False,False,False,False,False
5,nsw_price_asinh_lag_96,True,True,True,True,True,True,True,True,False,...,False,False,False,False,True,False,False,False,False,False
6,nsw_price_asinh_lag_336,True,True,True,True,True,True,True,True,False,...,False,False,False,False,True,False,False,False,False,False
7,nsw_price_asinh_lag_337,True,True,True,True,True,True,True,True,False,...,False,False,False,False,True,False,False,False,False,False
8,nsw_price_asinh_rmean_48,False,False,False,False,False,False,False,True,False,...,False,False,False,False,True,False,False,False,False,False
9,nsw_price_asinh_rmean_336,True,True,True,False,False,True,True,True,False,...,False,False,False,False,True,False,False,False,False,False


In [11]:
data = []
horizon_list = list(range(1,variables.HORIZON_COUNT +1))

for horizon in horizon_list:
    h = f"h{horizon}"

    features_for_this_horizon = features_optimal_amount.loc[features_optimal_amount[h] == True, "feature"].tolist()

    X_train_h = X_train[features_for_this_horizon]
    X_validate_h = X_validate[features_for_this_horizon] 
    X_test_h = X_test[features_for_this_horizon]

    for model_name, hyperparameters, y_train_df, y_validate_df, loss_weights_df in model_specs:
        data.append({
            "horizon": horizon,
            "model_name": model_name,
            "hyperparameters": hyperparameters,
            "X_train": X_train_h,
            "y_train": y_train_df[h],
            "X_validate": X_validate_h,
            "y_validate": y_validate_df[h],
            "loss_weights": loss_weights_df[h],
        })

In [12]:
print("model count:",variables.HORIZON_COUNT * 8)

model count: 768


In [13]:
trained_models = run_parallel(function=_train_model, data=data)

os=windows cpu_count=8 max_assigned_workers=1


Processing..: 100%|██████████| 768/768 [00:06<00:00, 116.22item/s]


Export

In [14]:
for m in trained_models:
    filename = f"h{m['horizon']:02d}_{m['model_name']}.joblib"
    joblib.dump(m["model"], f"{variables.TRAINED_MODELS_PATH}/{filename}")